# PathSnap: AdaBoost on Kaggle

Run the same 100 depth-one trees, learning rate 0.5, seed 42 and five existing trip folds as the local experiment. This notebook runs AdaBoost only, saves its fitted pipeline, and evaluates student-testing for reporting. Final selection remains based on cross-validation after all five models finish.

1. Upload `pathsnap_adaboost.zip` as a **private Kaggle dataset** and attach it to this notebook.
2. Enable Internet for the dependency installation cell. Use the CPU runtime: this scikit-learn tree-based implementation does not use the GPU.
3. Run all cells, then download `adaboost_results.zip` from the notebook output.

Do not add evaluator-testing data. GPU limitation: https://scikit-learn.org/stable/faq.html#will-you-add-gpu-support


In [ ]:
import os
import sys
import shutil
from pathlib import Path
from zipfile import ZipFile

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['MPLCONFIGDIR'] = '/kaggle/working/matplotlib-cache'

def prepare_bundle(input_root, work):
    input_root, work = Path(input_root), Path(work)
    work.mkdir(parents=True, exist_ok=True)
    # Kaggle normally expands uploaded ZIP datasets into their original folders.
    roots = [p.parent for p in input_root.rglob('requirements-kaggle.txt')
             if (p.parent / 'features/feature_schema.json').is_file()
             and (p.parent / 'src/pathsnap/train.py').is_file()]
    if len(roots) == 1:
        shutil.copytree(roots[0], work, dirs_exist_ok=True)
    elif len(roots) > 1:
        raise ValueError('Attach only one PathSnap feature dataset')
    else:
        archives = list(input_root.rglob('pathsnap_adaboost.zip'))
        if len(archives) != 1:
            raise ValueError('Attach the uploaded AdaBoost dataset using Add Input')
        with ZipFile(archives[0]) as archive:
            archive.extractall(work)
    return work

work = prepare_bundle('/kaggle/input', '/kaggle/working/pathsnap_adaboost')
print('Bundle ready at', work)


In [ ]:
import subprocess

# Install before importing NumPy, pandas or scikit-learn in this kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(work / 'requirements-kaggle.txt')], check=True)
sys.path.insert(0, str(work / 'src'))


In [ ]:
from pathsnap.train import parser, run

output = Path('/kaggle/working/adaboost_run')
arguments = [
    '--feature-dir', str(work / 'features'),
    '--output', str(output),
    '--models', 'adaboost', '--baseline-only',
    '--seed', '42', '--jobs', '1',
]
if (output / 'run.json').exists():
    arguments.append('--resume')
comparison = run(parser().parse_args(arguments))
display(comparison)


In [ ]:
import shutil
from IPython.display import FileLink, display

archive = shutil.make_archive('/kaggle/working/adaboost_results', 'zip', root_dir=output)
display(FileLink(archive))
print('Download adaboost_results.zip and import it into the local run using docs/kaggle.md.')


## Keep the experiment comparable

The bundle contains all four Parquet tables and the original feature schema. It preserves excluded trips and unresolved labels. Candidate miss rate includes zero-candidate observations with valid references. Input hashes and environment versions are saved with the results.

Training and inference times from different machines are not directly comparable. Use a common machine for final latency comparisons and timing tie breakers. Kaggle student-testing scores must not choose the winner; that decision uses the saved cross-validation scores.
